<a href="https://colab.research.google.com/github/kimheeseo/LSCNS/blob/main/GSNR_Calculator_for_Subsea_Open_Cables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
GSNR Calculator for Subsea Open Cables
Based on: "Design, Acceptance and Capacity of Subsea Open Cables"
Rivera Hartling et al., Journal of Lightwave Technology, Vol. 39, No. 3, Feb. 2021

Key equations implemented:
  - Design OSNR          (Eq. 1)
  - SNR_ASE scaling      (Eq. 5)
  - GSNR scaling         (Eq. 6)
  - SNR_GAWBS            (Eq. 7)
  - Generalized Droop    (Eq. 10)
  - GSNR from modem      (Eq. 11)
  - Multi-segment        (Eq. 22, 23)
  - Shannon capacity     (Eq. 19)
"""

import math
from dataclasses import dataclass, field
from typing import Optional


# ─────────────────────────────────────────────
# Data classes
# ─────────────────────────────────────────────

@dataclass
class SystemParams:
    """Core wet-plant parameters."""
    # OSNR design equation (Eq. 1)
    ptop_dBm: float        # EDFA total output power [dBm]
    n_channels: int        # Number of WDM channels
    gain_dB: float         # Repeater gain [dB]
    noise_figure_dB: float # Repeater noise figure [dB]
    n_repeaters: int       # Number of repeaters

    # Scaling / spectral parameters
    bo_GHz: float = 12.5      # Optical noise bandwidth [GHz] (default 12.5 GHz @ 1550 nm)
    delta_f_GHz: float = 37.5 # Channel spacing [GHz]
    chi: float = 0.95          # Bandwidth occupancy Be/Δf (typically > 0.9)

    # GAWBS
    fiber_aeff_um2: float = 150.0  # Fiber effective area [µm²]
    distance_Mm: float = 10.0      # Total transmission distance [Mm]

    # Nonlinear SNR (optional; if None, NLI is ignored)
    snr_nli_dB: Optional[float] = None

    # Modem implementation penalty
    snr_modem_dB: Optional[float] = None  # SNRm  (back-to-back implementation noise)
    snr_i_dB: Optional[float] = None      # SNRi  (propagation-specific modem penalties)


@dataclass
class SegmentGSNR:
    """GSNR result for one cable segment."""
    gsnr_dB: float
    snrase_dB: float
    snr_gawbs_dB: float
    snr_nli_dB: Optional[float]


# ─────────────────────────────────────────────
# Helper conversions
# ─────────────────────────────────────────────

def dB_to_linear(x_dB: float) -> float:
    return 10 ** (x_dB / 10)

def linear_to_dB(x_lin: float) -> float:
    return 10 * math.log10(x_lin)

def add_snr_in_dB(*snr_dB_values: float) -> float:
    """Combine multiple SNR contributions: 1/SNR_total = Σ 1/SNR_i."""
    total_inv = sum(1.0 / dB_to_linear(v) for v in snr_dB_values)
    return linear_to_dB(1.0 / total_inv)


# ─────────────────────────────────────────────
# Core calculation functions
# ─────────────────────────────────────────────

def design_osnr_dB(params: SystemParams) -> float:
    """
    Eq. 1 – Design OSNR (average, per channel, in dB/0.1 nm).
      OSNR_Design = 58 + P_TOP - N_Ch - G - NF - N_R
    All quantities in dB / dBm.
    """
    n_ch_dB  = 10 * math.log10(params.n_channels)
    n_rep_dB = 10 * math.log10(params.n_repeaters)
    return (58
            + params.ptop_dBm
            - n_ch_dB
            - params.gain_dB
            - params.noise_figure_dB
            - n_rep_dB)


def snrase_from_osnr_dB(osnr_dB: float,
                         bo_GHz: float,
                         delta_f_GHz: float) -> float:
    """
    Eq. 5 – Convert OSNR [dB/0.1 nm] → SNR_ASE [dB] (pure ratio).
      SNR_ASE = (B0 / Δf) × OSNR_ASE   (linear)
    """
    osnr_lin   = dB_to_linear(osnr_dB)
    snrase_lin = (bo_GHz / delta_f_GHz) * osnr_lin
    return linear_to_dB(snrase_lin)


def snr_gawbs_dB(fiber_aeff_um2: float, distance_Mm: float) -> float:
    """
    Eq. 7 – GAWBS SNR contribution.
      SNR_GAWBS = 1 / (Γ_GAWBS × L)

    Γ_GAWBS values from Table in Section II-C [dB/Mm]:
      150 µm² → -30.2 dB/Mm
      130 µm² → -29.6 dB/Mm
      110 µm² → -28.6 dB/Mm
       80 µm² → -27.5 dB/Mm
    """
    gamma_table = {150: -30.2, 130: -29.6, 110: -28.6, 80: -27.5}

    # Find nearest known effective area
    closest_aeff = min(gamma_table.keys(), key=lambda a: abs(a - fiber_aeff_um2))
    gamma_dB_per_Mm = gamma_table[closest_aeff]

    gamma_lin = dB_to_linear(gamma_dB_per_Mm)          # [linear/Mm]
    snr_gawbs_lin = 1.0 / (gamma_lin * distance_Mm)
    return linear_to_dB(snr_gawbs_lin)


def generalized_droop_gsnr_dB(snrase_dB: float,
                                snr_nli_dB: Optional[float],
                                snr_gawbs_dB_val: float) -> float:
    """
    Eq. 10 – Generalized Droop formula.
      (1 + 1/GSNR) = (1 + 1/SNR_ASE)(1 + 1/SNR_NLI)(1 + 1/SNR_GAWBS)

    If snr_nli_dB is None, the NLI term is omitted (linear-only regime).
    """
    snrase_lin   = dB_to_linear(snrase_dB)
    snrgawbs_lin = dB_to_linear(snr_gawbs_dB_val)

    product = (1 + 1 / snrase_lin) * (1 + 1 / snrgawbs_lin)

    if snr_nli_dB is not None:
        snrnli_lin = dB_to_linear(snr_nli_dB)
        product   *= (1 + 1 / snrnli_lin)

    gsnr_lin = 1.0 / (product - 1)
    return linear_to_dB(gsnr_lin)


def gsnr_from_modem_dB(snr_ext_dB: float, snr_i_dB: float) -> float:
    """
    Eq. 11 – Extract wet-plant GSNR from measured modem SNR_EXT.
      1/GSNR = 1/SNR_EXT - 1/SNR_i
    """
    inv_gsnr = 1 / dB_to_linear(snr_ext_dB) - 1 / dB_to_linear(snr_i_dB)
    if inv_gsnr <= 0:
        raise ValueError("SNR_EXT must be lower than SNR_i (modem penalty).")
    return linear_to_dB(1.0 / inv_gsnr)


def gsnr_scale_from_gosnr_dB(gosnr_dB: float,
                               bo_GHz: float,
                               chi: float,
                               delta_f_GHz: float) -> float:
    """
    Eq. 6 – Scale GOSNR → GSNR.
      GSNR = (B0 / (χ × Δf)) × GOSNR
    """
    gosnr_lin = dB_to_linear(gosnr_dB)
    gsnr_lin  = (bo_GHz / (chi * delta_f_GHz)) * gosnr_lin
    return linear_to_dB(gsnr_lin)


def multisegment_gsnr_dB(segment_gsnr_dB_list: list[float]) -> float:
    """
    Eq. 23 – Concatenate GSNR across multiple segments.
      1/GSNR_E2E = Σ 1/GSNR_k
    """
    total_inv = sum(1.0 / dB_to_linear(g) for g in segment_gsnr_dB_list)
    return linear_to_dB(1.0 / total_inv)


def shannon_capacity_Tbps(gsnr_per_channel_dB: list[float],
                           n_fiber_pairs: int,
                           bandwidth_THz: float) -> float:
    """
    Eq. 19/20 – Shannon capacity [Tbps] for a whole cable.
      C = 2 × N_FP × B × Σ log2(1 + GSNR_j)
    Simplified: uniform GSNR across bandwidth channels.
    """
    capacity = 0.0
    for gsnr_dB in gsnr_per_channel_dB:
        gsnr_lin  = dB_to_linear(gsnr_dB)
        capacity += math.log2(1 + gsnr_lin)

    # 2× for dual polarisation, scale to Tbps
    n_bins    = len(gsnr_per_channel_dB)
    bin_bw_Hz = (bandwidth_THz * 1e12) / n_bins
    capacity_bps = 2 * n_fiber_pairs * bandwidth_THz * 1e12 * (capacity / n_bins)
    return capacity_bps / 1e12


# ─────────────────────────────────────────────
# High-level API
# ─────────────────────────────────────────────

def calculate_gsnr(params: SystemParams,
                   verbose: bool = True) -> SegmentGSNR:
    """
    Full GSNR calculation pipeline for one cable segment.

    Steps
    -----
    1. Design OSNR             (Eq. 1)
    2. SNR_ASE from OSNR       (Eq. 5)
    3. SNR_GAWBS               (Eq. 7)
    4. Generalized Droop GSNR  (Eq. 10)
    """
    # Step 1 – Design OSNR
    osnr_dB = design_osnr_dB(params)

    # Step 2 – SNR_ASE
    snrase_dB = snrase_from_osnr_dB(osnr_dB, params.bo_GHz, params.delta_f_GHz)

    # Step 3 – SNR_GAWBS
    snr_gawbs = snr_gawbs_dB(params.fiber_aeff_um2, params.distance_Mm)

    # Step 4 – GSNR (generalized droop)
    gsnr_dB = generalized_droop_gsnr_dB(snrase_dB, params.snr_nli_dB, snr_gawbs)

    result = SegmentGSNR(
        gsnr_dB    = gsnr_dB,
        snrase_dB  = snrase_dB,
        snr_gawbs_dB = snr_gawbs,
        snr_nli_dB   = params.snr_nli_dB,
    )

    if verbose:
        _print_results(params, osnr_dB, result)

    return result


def _print_results(params: SystemParams,
                   osnr_dB: float,
                   result: SegmentGSNR) -> None:
    sep = "─" * 52
    print(f"\n{'═'*52}")
    print(f"  GSNR Calculator — Subsea Open Cable")
    print(f"{'═'*52}")
    print(f"\n{'Input Parameters':}")
    print(sep)
    print(f"  Total output power (P_TOP)  : {params.ptop_dBm:>8.2f} dBm")
    print(f"  Number of channels          : {params.n_channels:>8d}")
    print(f"  Repeater gain               : {params.gain_dB:>8.2f} dB")
    print(f"  Noise figure                : {params.noise_figure_dB:>8.2f} dB")
    print(f"  Number of repeaters         : {params.n_repeaters:>8d}")
    print(f"  Channel spacing (Δf)        : {params.delta_f_GHz:>8.2f} GHz")
    print(f"  Bandwidth occupancy (χ)     : {params.chi:>8.3f}")
    print(f"  Fiber A_eff                 : {params.fiber_aeff_um2:>8.1f} µm²")
    print(f"  Cable length                : {params.distance_Mm:>8.2f} Mm")
    if params.snr_nli_dB:
        print(f"  SNR_NLI                     : {params.snr_nli_dB:>8.2f} dB")

    print(f"\n{'Results':}")
    print(sep)
    print(f"  Design OSNR                 : {osnr_dB:>8.2f} dB/0.1 nm  (Eq. 1)")
    print(f"  SNR_ASE                     : {result.snrase_dB:>8.2f} dB          (Eq. 5)")
    print(f"  SNR_GAWBS                   : {result.snr_gawbs_dB:>8.2f} dB          (Eq. 7)")
    if result.snr_nli_dB:
        print(f"  SNR_NLI                     : {result.snr_nli_dB:>8.2f} dB")
    print(sep)
    print(f"  ★  GSNR (Generalized Droop) : {result.gsnr_dB:>8.2f} dB          (Eq. 10)")
    print(f"{'═'*52}\n")


# ─────────────────────────────────────────────
# Demo / example usage
# ─────────────────────────────────────────────

if __name__ == "__main__":

    # ── Example 1: Single-segment trans-Pacific system ──────────────────
    print("\n[ Example 1 ] Trans-Pacific single-segment system")
    params_1 = SystemParams(
        ptop_dBm        = 17.0,   # dBm
        n_channels      = 96,
        gain_dB         = 13.0,   # dB (≈ span loss)
        noise_figure_dB = 5.5,    # dB
        n_repeaters     = 100,
        bo_GHz          = 12.5,
        delta_f_GHz     = 37.5,
        chi             = 0.95,
        fiber_aeff_um2  = 150.0,
        distance_Mm     = 9.0,    # ~9,000 km
        snr_nli_dB      = 22.0,   # dB
    )
    result_1 = calculate_gsnr(params_1)

    # ── Example 2: Short Atlantic system ────────────────────────────────
    print("\n[ Example 2 ] Short Atlantic system (lower NLI)")
    params_2 = SystemParams(
        ptop_dBm        = 16.0,
        n_channels      = 120,
        gain_dB         = 12.5,
        noise_figure_dB = 5.0,
        n_repeaters     = 60,
        bo_GHz          = 12.5,
        delta_f_GHz     = 37.5,
        chi             = 0.95,
        fiber_aeff_um2  = 150.0,
        distance_Mm     = 6.0,
        snr_nli_dB      = 25.0,
    )
    result_2 = calculate_gsnr(params_2)

    # ── Example 3: Multi-segment concatenation ───────────────────────────
    print("\n[ Example 3 ] Multi-segment concatenation (Eq. 23)")
    segments = [result_1.gsnr_dB, result_2.gsnr_dB]
    gsnr_e2e = multisegment_gsnr_dB(segments)
    print(f"  Segment 1 GSNR : {segments[0]:.2f} dB")
    print(f"  Segment 2 GSNR : {segments[1]:.2f} dB")
    print(f"  ★  E2E GSNR    : {gsnr_e2e:.2f} dB\n")

    # ── Example 4: Shannon capacity estimate ────────────────────────────
    print("[ Example 4 ] Shannon capacity estimate (Eq. 20)")
    n_fp           = 8           # fiber pairs
    bandwidth_THz  = 4.5         # C-band
    # Uniform GSNR across 96 channels
    gsnr_channels  = [result_1.gsnr_dB] * 96
    capacity       = shannon_capacity_Tbps(gsnr_channels, n_fp, bandwidth_THz)
    print(f"  GSNR (uniform) : {result_1.gsnr_dB:.2f} dB")
    print(f"  Fiber pairs    : {n_fp}")
    print(f"  Bandwidth      : {bandwidth_THz} THz")
    print(f"  ★  Capacity    : {capacity:.1f} Tbps\n")

    # ── Example 5: GSNR from modem measurement (Eq. 11) ─────────────────
    print("[ Example 5 ] GSNR extracted from modem SNR_EXT (Eq. 11)")
    snr_ext_dB = 14.5   # measured after transmission
    snr_i_dB   = 30.0   # modem implementation penalty
    gsnr_modem = gsnr_from_modem_dB(snr_ext_dB, snr_i_dB)
    print(f"  SNR_EXT (measured) : {snr_ext_dB:.2f} dB")
    print(f"  SNR_i (modem)      : {snr_i_dB:.2f} dB")
    print(f"  ★  GSNR            : {gsnr_modem:.2f} dB\n")


[ Example 1 ] Trans-Pacific single-segment system

════════════════════════════════════════════════════
  GSNR Calculator — Subsea Open Cable
════════════════════════════════════════════════════

Input Parameters
────────────────────────────────────────────────────
  Total output power (P_TOP)  :    17.00 dBm
  Number of channels          :       96
  Repeater gain               :    13.00 dB
  Noise figure                :     5.50 dB
  Number of repeaters         :      100
  Channel spacing (Δf)        :    37.50 GHz
  Bandwidth occupancy (χ)     :    0.950
  Fiber A_eff                 :    150.0 µm²
  Cable length                :     9.00 Mm
  SNR_NLI                     :    22.00 dB

Results
────────────────────────────────────────────────────
  Design OSNR                 :    16.68 dB/0.1 nm  (Eq. 1)
  SNR_ASE                     :    11.91 dB          (Eq. 5)
  SNR_GAWBS                   :    20.66 dB          (Eq. 7)
  SNR_NLI                     :    22.00 dB
───────────